In [ ]:
from pyspark.sql import functions as F

# Read existing Gold tables
dim_date = spark.read.table("DimDate")
dim_gdp = spark.read.table("DimGDP")

# Collect distinct years from both tables
date_years = dim_date.select(
    F.col("year").cast("int").alias("year")
)

gdp_years = dim_gdp.select(
    F.col("year").cast("int").alias("year")
)

# Create a unified Year dimension
dim_year = (
    date_years
    .unionByName(gdp_years)
    .filter(F.col("year").isNotNull())
    .distinct()
    .withColumn("year_key", F.col("year"))
    .select("year_key", "year")
    .orderBy("year")
)

# Display results
print("DimYear rows:", dim_year.count())
display(dim_year)

# Save to Gold Lakehouse
(
    dim_year.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("DimYear")
)

# Verify saved table
final_dim_year = spark.read.table("DimYear")

print("Verified DimYear rows:", final_dim_year.count())
display(final_dim_year)

StatementMeta(, 68c88886-3059-468e-bbc5-c75ef1fc11b4, 3, Finished, Available, Finished, False)

DimYear rows: 67


SynapseWidget(Synapse.DataFrame, ad76cede-2a09-4e63-bb22-e239bd48e3bb)

Verified DimYear rows: 67


SynapseWidget(Synapse.DataFrame, e52d7f2d-b019-46f7-8c16-ce1190709aa4)